In [1]:
import os
os.chdir('/Users/taruni/Desktop/virality-prediction-ml-2026')
print('Working directory:', os.getcwd())

Working directory: /Users/taruni/Desktop/virality-prediction-ml-2026


# 02 — Preprocessing & Feature Engineering

In [2]:
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
print('TODO: clean data and engineer features')

TODO: clean data and engineer features


In [3]:
# ── STEP 1: LOAD ALL 3 DATASETS ──
instagram_df = pd.read_csv('data/raw/Instagram_Analytics.csv')
youtube_df   = pd.read_csv('data/raw/USvideos.csv', encoding='latin-1')
tiktok_df    = pd.read_csv('data/raw/tiktok_dataset.csv')

# Drop TikTok missing rows
tiktok_df = tiktok_df.dropna().reset_index(drop=True)

print('Instagram:', instagram_df.shape)
print('YouTube:  ', youtube_df.shape)
print('TikTok:   ', tiktok_df.shape)

Instagram: (29999, 23)
YouTube:   (40949, 16)
TikTok:    (19084, 12)


In [4]:
# ── STEP 2: COMPUTE VIRALITY LABELS ──

# Instagram
ig_threshold = instagram_df['engagement_rate'].quantile(0.80)
instagram_df['viral'] = (instagram_df['engagement_rate'] > ig_threshold).astype(int)

# YouTube
youtube_df['engagement_rate'] = (youtube_df['likes'] + youtube_df['comment_count']) / youtube_df['views']
yt_threshold = youtube_df['engagement_rate'].quantile(0.80)
youtube_df['viral'] = (youtube_df['engagement_rate'] > yt_threshold).astype(int)

# TikTok
tiktok_df['engagement_rate'] = (tiktok_df['video_like_count'] + tiktok_df['video_comment_count']) / tiktok_df['video_view_count']
tt_threshold = tiktok_df['engagement_rate'].quantile(0.80)
tiktok_df['viral'] = (tiktok_df['engagement_rate'] > tt_threshold).astype(int)

print('Virality labels created!')
print(f'Instagram viral %: {instagram_df["viral"].mean():.1%}')
print(f'YouTube viral %:   {youtube_df["viral"].mean():.1%}')
print(f'TikTok viral %:    {tiktok_df["viral"].mean():.1%}')

Virality labels created!
Instagram viral %: 19.9%
YouTube viral %:   20.0%
TikTok viral %:    20.0%


In [5]:
# ── STEP 3: FEATURE ENGINEERING ──
analyzer = SentimentIntensityAnalyzer()

# ── INSTAGRAM ── already has most features!
instagram_df['platform'] = 'instagram'
instagram_df['sentiment'] = 0.0  # no caption text available in this dataset

# ── YOUTUBE ──
youtube_df['platform'] = 'youtube'

# Extract hour and day from publish_time
youtube_df['publish_time'] = pd.to_datetime(youtube_df['publish_time'])
youtube_df['post_hour']   = youtube_df['publish_time'].dt.hour
youtube_df['day_of_week'] = youtube_df['publish_time'].dt.dayofweek

# Sentiment from title + tags
def get_sentiment(text):
    if pd.isna(text) or text == '':
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

youtube_df['caption_text']   = youtube_df['title'].fillna('') + ' ' + youtube_df['tags'].fillna('')
youtube_df['sentiment']      = youtube_df['caption_text'].apply(get_sentiment)
youtube_df['caption_length'] = youtube_df['title'].fillna('').apply(len)
youtube_df['hashtags_count'] = youtube_df['tags'].fillna('').apply(lambda x: len(str(x).split('|')))

# Map category_id to name
import json
with open('data/raw/US_category_id.json') as f:
    cat_data = json.load(f)
cat_map = {int(item['id']): item['snippet']['title'] for item in cat_data['items']}
youtube_df['content_category'] = youtube_df['category_id'].map(cat_map)

# ── TIKTOK ──
tiktok_df['platform'] = 'tiktok'
tiktok_df['sentiment'] = tiktok_df['video_transcription_text'].apply(get_sentiment)
tiktok_df['caption_length'] = tiktok_df['video_transcription_text'].fillna('').apply(len)
tiktok_df['post_hour']   = 12  # not available, use neutral default
tiktok_df['day_of_week'] = 0   # not available, use neutral default
tiktok_df['hashtags_count'] = 0  # not available
tiktok_df['content_category'] = 'Unknown'

print('Feature engineering done!')
print('Instagram columns:', instagram_df.columns.tolist())

Feature engineering done!
Instagram columns: ['post_id', 'account_id', 'account_type', 'follower_count', 'media_type', 'content_category', 'traffic_source', 'has_call_to_action', 'post_datetime', 'post_date', 'post_hour', 'day_of_week', 'likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate', 'followers_gained', 'caption_length', 'hashtags_count', 'performance_bucket_label', 'viral', 'platform', 'sentiment']


In [6]:
# ── STEP 4: SELECT FEATURES & MERGE ──

# Define the common feature columns we want from each platform
FEATURES = ['platform', 'sentiment', 'hashtags_count', 'post_hour', 
            'day_of_week', 'caption_length', 'content_category', 
            'engagement_rate', 'viral']

# Instagram — select and rename
ig = instagram_df[FEATURES].copy()

# YouTube — select and rename
yt = youtube_df[FEATURES].copy()

# TikTok — select and rename
tt = tiktok_df[FEATURES].copy()

# Merge all 3
master_df = pd.concat([ig, yt, tt], ignore_index=True)

print('Master dataset shape:', master_df.shape)
print()
print('Platform counts:')
print(master_df['platform'].value_counts())
print()
print('Viral counts:')
print(master_df['viral'].value_counts())
print()
print('Missing values:')
print(master_df.isnull().sum())

Master dataset shape: (90032, 9)

Platform counts:
platform
youtube      40949
instagram    29999
tiktok       19084
Name: count, dtype: int64

Viral counts:
viral
0    72061
1    17971
Name: count, dtype: int64

Missing values:
platform            0
sentiment           0
hashtags_count      0
post_hour           0
day_of_week         0
caption_length      0
content_category    0
engagement_rate     0
viral               0
dtype: int64


In [7]:
# ── STEP 5: ENCODE CATEGORICALS & SAVE ──

# One-hot encode platform and content_category
master_encoded = pd.get_dummies(master_df, columns=['platform', 'content_category'], drop_first=False)

print('Shape after encoding:', master_encoded.shape)
print()
print('Columns:', master_encoded.columns.tolist())

# Save to processed folder
master_encoded.to_csv('data/processed/master.csv', index=False)
print()
print('Saved to data/processed/master.csv!')

Shape after encoding: (90032, 35)

Columns: ['sentiment', 'hashtags_count', 'post_hour', 'day_of_week', 'caption_length', 'engagement_rate', 'viral', 'platform_instagram', 'platform_tiktok', 'platform_youtube', 'content_category_Autos & Vehicles', 'content_category_Beauty', 'content_category_Comedy', 'content_category_Education', 'content_category_Entertainment', 'content_category_Fashion', 'content_category_Film & Animation', 'content_category_Fitness', 'content_category_Food', 'content_category_Gaming', 'content_category_Howto & Style', 'content_category_Lifestyle', 'content_category_Music', 'content_category_News & Politics', 'content_category_Nonprofits & Activism', 'content_category_People & Blogs', 'content_category_Pets & Animals', 'content_category_Photography', 'content_category_Science & Technology', 'content_category_Shows', 'content_category_Sports', 'content_category_Technology', 'content_category_Travel', 'content_category_Travel & Events', 'content_category_Unknown']

Sa